# 4상태 축소 모델 — 손계산 교차 검증

MarkovBoard의 43상태 모델로 가기 전에, **손으로 풀 수 있는 크기**의 장난감 보드를
하나 만들어 손계산 · NumPy · TypeScript 세 경로가 같은 답을 내는지 확인한다.
세 결과가 일치해야 실제 모델로 진행한다. (계획서 Phase 2 · §11.3)

## 장난감 보드의 규칙

- 칸은 0 · 1 · 2 · 3 네 개이며 순환한다.
- 주사위는 한 개, 눈은 **1과 2뿐**이고 각각 확률 1/2. 더블 규칙은 없다.
- **3번 칸에 착지하면 즉시 1번으로 끌려간다.**

세 번째 규칙이 이 보드를 고른 이유다. 실제 모델의 '무인도로 가시오' 카드와
12번 황금열쇠 '뒤로 2칸'이 하는 일이 똑같고, 그 결과 **3번에서 턴을 시작하는 일이
없어진다.** BFS 도달성 검사(§3.2)가 무엇을 걸러내는지 보여주는 최소 예제다.

## 1. 전이행렬을 손으로 쓴다

각 칸에서 +1 또는 +2를 확률 1/2씩으로 이동하고, 3번에 닿으면 1번으로 보낸다.

| 출발 | +1 (1/2) | +2 (1/2) |
|---|---|---|
| 0 | 1 | 2 |
| 1 | 2 | 3 → **1** |
| 2 | 3 → **1** | 0 |
| 3 | 0 | 1 |

$$
P=\begin{pmatrix}
0 & 1/2 & 1/2 & 0\\
0 & 1/2 & 1/2 & 0\\
1/2 & 1/2 & 0 & 0\\
1/2 & 1/2 & 0 & 0
\end{pmatrix}
$$

**3번 열이 전부 0**이다. 즉 어떤 상태에서도 3번으로 갈 수 없다.

In [1]:
import numpy as np
from fractions import Fraction

def build_toy_matrix(size=4, faces=(1, 2), forced={3: 1}):
    """장난감 보드의 전이행렬. TypeScript 쪽 buildToyMatrix() 와 같은 규칙이다."""
    p = Fraction(1, len(faces))
    P = [[Fraction(0)] * size for _ in range(size)]
    for i in range(size):
        for f in faces:
            j = forced.get((i + f) % size, (i + f) % size)
            P[i][j] += p
    return P

P = build_toy_matrix()
for row in P:
    print([str(x) for x in row], " 행 합 =", sum(row))

['0', '1/2', '1/2', '0']  행 합 = 1
['0', '1/2', '1/2', '0']  행 합 = 1
['1/2', '1/2', '0', '0']  행 합 = 1
['1/2', '1/2', '0', '0']  행 합 = 1


## 2. 도달 불가능 상태를 제거한다

출발 상태 0에서 BFS로 도달 가능한 상태를 찾는다.

In [2]:
from collections import deque

def reachable(P, start=0):
    seen, queue = {start}, deque([start])
    while queue:
        i = queue.popleft()
        for j, p in enumerate(P[i]):
            if p > 0 and j not in seen:
                seen.add(j)
                queue.append(j)
    return sorted(seen)

states = reachable(P)
print("도달 가능 상태:", states)
print("제거된 상태  :", [i for i in range(4) if i not in states])

Q = [[P[i][j] for j in states] for i in states]
for i, row in zip(states, Q):
    print(i, [str(x) for x in row])

도달 가능 상태: [0, 1, 2]
제거된 상태  : [3]
0 ['0', '1/2', '1/2']
1 ['0', '1/2', '1/2']
2 ['1/2', '1/2', '0']


## 3. 정상분포를 손으로 푼다

$\pi P = \pi$ 를 성분별로 쓰면 (열 = 도착 상태)

$$
\begin{aligned}
\pi_0 &= \tfrac12 \pi_2\\
\pi_1 &= \tfrac12\pi_0 + \tfrac12\pi_1 + \tfrac12\pi_2\\
\pi_2 &= \tfrac12\pi_0 + \tfrac12\pi_1
\end{aligned}
$$

첫 식에서 $\pi_0 = \pi_2/2$. 이를 셋째 식에 넣으면

$$\pi_2 = \tfrac14\pi_2 + \tfrac12\pi_1 \;\Longrightarrow\; \pi_1 = \tfrac32 \pi_2$$

정규화 조건 $\pi_0+\pi_1+\pi_2 = 1$ 에 대입하면

$$\tfrac12\pi_2 + \tfrac32\pi_2 + \pi_2 = 3\pi_2 = 1$$

$$\boxed{\;\pi = \left(\tfrac16,\ \tfrac12,\ \tfrac13\right)\;}$$

둘째 식은 자동으로 만족된다(확인용): $\tfrac1{12}+\tfrac14+\tfrac16 = \tfrac12$ ✓

In [3]:
# 손계산 결과
pi_hand = [Fraction(1, 6), Fraction(1, 2), Fraction(1, 3)]
print("손계산 π =", [str(x) for x in pi_hand], " 합 =", sum(pi_hand))

# πP = π 를 유리수로 정확히 검산한다 (부동소수점 오차 없음)
lhs = [sum(pi_hand[i] * Q[i][j] for i in range(3)) for j in range(3)]
print("πP      =", [str(x) for x in lhs])
print("일치     :", lhs == pi_hand)

손계산 π = ['1/6', '1/2', '1/3']  합 = 1
πP      = ['1/6', '1/2', '1/3']
일치     : True


## 4. NumPy로 같은 답이 나오는지 확인한다

두 가지 독립적인 경로로 구한다.

1. **고유벡터** — $P^{\mathsf T}$ 의 고유값 1에 대응하는 고유벡터를 정규화
2. **멱승법** — $\pi^{(k+1)} = \pi^{(k)}P$ 를 반복

계산 경로가 다르므로, 두 값이 손계산과 모두 맞으면 신뢰도가 크게 올라간다.

In [4]:
Qf = np.array([[float(x) for x in row] for row in Q])
pi_exact = np.array([float(x) for x in pi_hand])

# (1) 고유벡터
vals, vecs = np.linalg.eig(Qf.T)
k = int(np.argmin(np.abs(vals - 1)))
pi_eig = np.real(vecs[:, k])
pi_eig = pi_eig / pi_eig.sum()
print("고유벡터 π =", pi_eig)

# (2) 멱승법
pi_pow = np.array([1.0, 0.0, 0.0])
for _ in range(200):
    pi_pow = pi_pow @ Qf
    pi_pow /= pi_pow.sum()
print("멱승법   π =", pi_pow)

print()
print("손계산   π =", pi_exact)
print("고유벡터 오차 =", np.abs(pi_eig - pi_exact).sum())
print("멱승법   오차 =", np.abs(pi_pow - pi_exact).sum())

고유벡터 π = [0.16666667 0.5        0.33333333]
멱승법   π = [0.16666667 0.5        0.33333333]

손계산   π = [0.16666667 0.5        0.33333333]
고유벡터 오차 = 4.996003610813204e-16
멱승법   오차 = 2.7755575615628914e-17


## 5. 두 번째 고유값을 손으로 푼다

특성다항식을 전개하면

$$\det(P - \lambda I) = -\lambda^3 + \tfrac12\lambda^2 + \tfrac12\lambda
= -\lambda\left(\lambda - 1\right)\left(\lambda + \tfrac12\right)$$

$$\lambda \in \left\{1,\ -\tfrac12,\ 0\right\}
\qquad\Longrightarrow\qquad |\lambda_2| = \tfrac12$$

$|\lambda_2| = 1/2$ 이므로 **오차가 매 스텝 정확히 절반으로 줄어야 한다.**
이것이 실제로 그런지 확인하면 §3.7의 수렴 이론이 이 크기에서 맞는다는 증거가 된다.

In [5]:
print("NumPy 고유값 :", np.sort_complex(np.linalg.eigvals(Qf)))
print("손계산 고유값 : [-0.5  0.   1. ]")

# 오차가 정말 매 스텝 절반씩 줄어드는가
pi_k = np.array([1.0, 0.0, 0.0])
prev = None
print()
print(" k   ||π⁽ᵏ⁾ − π||₁      비(오차 감쇠율)")
for k in range(1, 13):
    pi_k = pi_k @ Qf
    err = np.abs(pi_k - pi_exact).sum()
    ratio = "" if prev is None else f"{err / prev:.6f}"
    print(f"{k:2d}   {err:.10f}   {ratio}")
    prev = err

NumPy 고유값 : [-5.00000000e-01+0.j -3.81639165e-17+0.j  1.00000000e+00+0.j]
손계산 고유값 : [-0.5  0.   1. ]

 k   ||π⁽ᵏ⁾ − π||₁      비(오차 감쇠율)
 1   0.3333333333   
 2   0.1666666667   0.500000
 3   0.0833333333   0.500000
 4   0.0416666667   0.500000
 5   0.0208333333   0.500000
 6   0.0104166667   0.500000
 7   0.0052083333   0.500000
 8   0.0026041667   0.500000
 9   0.0013020833   0.500000
10   0.0006510417   0.500000
11   0.0003255208   0.500000
12   0.0001627604   0.500000


감쇠율이 **0.5로 수렴**한다. 이론값 $|\lambda_2| = 1/2$ 과 일치한다.

## 6. Kac 보조정리 확인

$$m_{ii} = \frac{1}{\pi_i}
\qquad\Longrightarrow\qquad
m_{00} = 6,\quad m_{11} = 2,\quad m_{22} = 3$$

$\pi$ 를 구한 경로와 **완전히 독립적인** 몬테카를로로 재귀시간을 실측해 맞춰 본다.
(계획서 §11.4 — 세 번째 검증 축)

In [6]:
rng = np.random.default_rng(20260908)

def simulate_return_times(Qf, start, trials=200_000):
    """start 로 돌아오기까지 걸린 스텝 수의 표본평균."""
    n = Qf.shape[0]
    cum = np.cumsum(Qf, axis=1)
    total = 0
    for _ in range(trials):
        s, steps = start, 0
        while True:
            s = int(np.searchsorted(cum[s], rng.random()))
            steps += 1
            if s == start:
                break
        total += steps
    return total / trials

print(" i   실측 m_ii    이론 1/π_i")
for i in range(3):
    m = simulate_return_times(Qf, i)
    print(f" {i}   {m:8.4f}    {1 / pi_exact[i]:8.4f}")

 i   실측 m_ii    이론 1/π_i


 0     5.9993      6.0000


 1     1.9957      2.0000


 2     2.9990      3.0000


## 결론

| 경로 | π₀ | π₁ | π₂ | \|λ₂\| |
|---|---|---|---|---|
| 손계산 | 1/6 | 1/2 | 1/3 | 1/2 |
| NumPy 고유벡터 | ✅ | ✅ | ✅ | ✅ |
| NumPy 멱승법 | ✅ | ✅ | ✅ | ✅ (감쇠율 0.5) |
| 몬테카를로 (Kac) | ✅ | ✅ | ✅ | — |

네 경로가 모두 일치한다. **Phase 2 완료 조건(손계산 = NumPy) 충족.**

TypeScript 구현은 `src/markov/toy.ts` 에 같은 규칙으로 들어 있고,
Phase 5에서 멱승법이 완성되면 `TOY_EXACT_PI` 와 대조하는 테스트가 붙는다.
이로써 **손계산 · NumPy · TypeScript** 세 방법의 대조가 완결된다.